# Stage 3 — Base ASAG Preprocessing Pipeline

**Project:** Automatic Short Answer Grading using Semantic and Dependency Analysis  
**Repository:** `proposition-level-asag`  
**Current Stage:** Stage 3 — Base ASAG Preprocessing

---

### Purpose
This notebook demonstrates the **Stage 3 Preprocessing Component** for the Base ASAG system.

The preprocessing pipeline performs:
1. **Sentence Segmentation:** Identifies sentence boundaries without relying on the full dependency parser.
2. **Tokenization:** Breaks sentences into linguistic tokens.
3. **Basic Normalization:** Standardizes unicode characters and whitespace without destructive text alterations.
4. **Lemmatization:** Reduces inflected word forms to canonical dictionary base forms (e.g., *identifies* $\rightarrow$ *identify*).
5. **Part-of-Speech (POS) Tagging:** Assigns both coarse-grained Universal POS tags and detailed Penn Treebank tags.

> **CRITICAL ASAG PRESERVATION RULE:**  
> Stop words, negation terms (*not*, *cannot*), and punctuation are strictly **preserved**. Destructive stop-word removal is avoided because subsequent stages (Stage 4 Dependency Parsing, Stage 5 Dependency Triples, and Stage 8 Textual Entailment) require complete syntactic and logical context.

> **NOTICE: DEMO DATA — NOT RESEARCH DATA**  
> The examples used here are synthetic demonstration examples for pipeline verification only.

## 1. Imports and Setup

We import pandas for tabular formatting and the public preprocessing API from `base_asag.preprocessing`. The actual preprocessing engine resides in `base_asag/preprocessing/` and is imported directly without code duplication.

In [1]:
import sys
from pathlib import Path
import pandas as pd

# Ensure repository root is on sys.path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from base_asag.preprocessing import TextPreprocessor, normalize_text

# Initialize preprocessor using spaCy's en_core_web_sm
preprocessor = TextPreprocessor(model_name="en_core_web_sm")
print("TextPreprocessor initialized successfully.")
print(f"Underlying spaCy model: {preprocessor.model_name}")

TextPreprocessor initialized successfully.
Underlying spaCy model: en_core_web_sm


## 2. Process Reference Answer & Student Answer

We test preprocessing on the global primary key demonstration example:
- **Reference Answer:** *"A primary key uniquely identifies each record in a table."*
- **Student Answer:** *"A primary key uniquely identifies every row in a table."*

In [2]:
reference_text = "A primary key uniquely identifies each record in a table."
student_text = "A primary key uniquely identifies every row in a table."

# Process both answers into ProcessedDocument objects
ref_doc = preprocessor.process(reference_text)
stu_doc = preprocessor.process(student_text)

print(f'Reference Answer: "{ref_doc.original_text}"')
print(f"  Sentence Count: {ref_doc.sentence_count}")
print(f"  Token Count:    {ref_doc.sentences[0].token_count}")
print()
print(f'Student Answer:   "{stu_doc.original_text}"')
print(f"  Sentence Count: {stu_doc.sentence_count}")
print(f"  Token Count:    {stu_doc.sentences[0].token_count}")

Reference Answer: "A primary key uniquely identifies each record in a table."
  Sentence Count: 1
  Token Count:    11

Student Answer:   "A primary key uniquely identifies every row in a table."
  Sentence Count: 1
  Token Count:    11


## 3. Linguistic Token Inspection

Below is the detailed token-level breakdown for the reference answer, displaying:
- **Surface Token:** The exact token string.
- **Lemma:** The dictionary root form (e.g. *identifies* $\rightarrow$ *identify*).
- **Universal POS:** Coarse syntactic category (e.g. `DET`, `ADJ`, `NOUN`, `ADV`, `VERB`, `PUNCT`).
- **Detailed Tag:** Penn Treebank tag (e.g. `DT`, `JJ`, `NN`, `RB`, `VBZ`).
- **Stop Word & Punctuation Flags:** Contextual markers stored for downstream components.

In [3]:
def tokens_to_dataframe(processed_sent):
    rows = []
    for tok in processed_sent.tokens_info:
        rows.append({
            "Token": tok.text,
            "Lemma": tok.lemma,
            "Universal POS": tok.pos,
            "Detailed Tag": tok.tag,
            "Stop Word?": tok.is_stop,
            "Punctuation?": tok.is_punct,
        })
    return pd.DataFrame(rows)

print("--- Reference Answer Token Table ---")
df_ref = tokens_to_dataframe(ref_doc.sentences[0])
df_ref

--- Reference Answer Token Table ---


,Token,Lemma,Universal POS,Detailed Tag,Stop Word?,Punctuation?
0,A,a,DET,DT,True,False
1,primary,primary,ADJ,JJ,False,False
2,key,key,NOUN,NN,False,False
3,uniquely,uniquely,ADV,RB,False,False
4,identifies,identify,VERB,VBZ,False,False
5,each,each,DET,DT,True,False
6,record,record,NOUN,NN,False,False
7,in,in,ADP,IN,True,False
8,a,a,DET,DT,True,False
9,table,table,NOUN,NN,False,False


In [4]:
print("--- Student Answer Token Table ---")
df_stu = tokens_to_dataframe(stu_doc.sentences[0])
df_stu

--- Student Answer Token Table ---


,Token,Lemma,Universal POS,Detailed Tag,Stop Word?,Punctuation?
0,A,a,DET,DT,True,False
1,primary,primary,ADJ,JJ,False,False
2,key,key,NOUN,NN,False,False
3,uniquely,uniquely,ADV,RB,False,False
4,identifies,identify,VERB,VBZ,False,False
5,every,every,DET,DT,True,False
6,row,row,NOUN,NN,False,False
7,in,in,ADP,IN,True,False
8,a,a,DET,DT,True,False
9,table,table,NOUN,NN,False,False


## 4. Side-by-Side Comparison: Morphological vs. Lexical Variation

Let us examine what preprocessing achieves and what remains for subsequent stages:

In [5]:
df_compare = pd.DataFrame({
    "Ref Token": ref_doc.sentences[0].tokens,
    "Ref Lemma": ref_doc.sentences[0].lemmas,
    "Ref POS":   ref_doc.sentences[0].pos_tags,
    "Stu Token": stu_doc.sentences[0].tokens,
    "Stu Lemma": stu_doc.sentences[0].lemmas,
    "Stu POS":   stu_doc.sentences[0].pos_tags,
})
df_compare

,Ref Token,Ref Lemma,Ref POS,Stu Token,Stu Lemma,Stu POS
0,A,a,DET,A,a,DET
1,primary,primary,ADJ,primary,primary,ADJ
2,key,key,NOUN,key,key,NOUN
3,uniquely,uniquely,ADV,uniquely,uniquely,ADV
4,identifies,identify,VERB,identifies,identify,VERB
5,each,each,DET,every,every,DET
6,record,record,NOUN,row,row,NOUN
7,in,in,ADP,in,in,ADP
8,a,a,DET,a,a,DET
9,table,table,NOUN,table,table,NOUN


### Observations from Comparison:
1. **Morphological Normalization (Resolved in Stage 3):**
   - In both answers, the inflected verb *"identifies"* is correctly mapped to its base lemma *"identify"* with POS tag `VERB`.
2. **Lexical / Synonymy Divergence (Pending Stage 6):**
   - Reference has `['each', 'record']` with lemmas `['each', 'record']`.
   - Student has `['every', 'row']` with lemmas `['every', 'row']`.
   - Preprocessing faithfully preserves these distinct lemmas and POS tags (`DET` and `NOUN`). Resolving that *"record"* and *"row"* are semantic synonyms in database contexts is deferred to Stage 6 (WordNet / Semantic Relations).
3. **Syntactic Relations (Pending Stage 4):**
   - The dependency relations linking the subject (*"primary key"*) to the predicate (*"identifies"*) and direct object (*"record"* / *"row"*) will be extracted in Stage 4 (Dependency Parsing).

## 5. Multi-Sentence & Negation Preservation

In ASAG answers, student responses frequently contain multiple sentences, negations, or extra clauses (e.g. Student B, C, D from Section 7 of `GEMINI.md`).
The preprocessor cleanly segments multiple sentences and preserves negation tokens without loss.

In [6]:
multi_sentence_example = (
    "A primary key uniquely identifies each record in a table. "
    "It cannot contain NULL values."
)

multi_doc = preprocessor.process(multi_sentence_example)

print(f"Segmented Sentences: {multi_doc.sentence_count}\n")
for i, sent in enumerate(multi_doc.sentences):
    print(f'Sentence {i+1}: "{sent.original_text}"')
    print(f"  Tokens: {sent.tokens}")
    print(f"  Lemmas: {sent.lemmas}")
    print(f"  POS:    {sent.pos_tags}")
    print()

Segmented Sentences: 2

Sentence 1: "A primary key uniquely identifies each record in a table."
  Tokens: ['A', 'primary', 'key', 'uniquely', 'identifies', 'each', 'record', 'in', 'a', 'table', '.']
  Lemmas: ['a', 'primary', 'key', 'uniquely', 'identify', 'each', 'record', 'in', 'a', 'table', '.']
  POS:    ['DET', 'ADJ', 'NOUN', 'ADV', 'VERB', 'DET', 'NOUN', 'ADP', 'DET', 'NOUN', 'PUNCT']

Sentence 2: "It cannot contain NULL values."
  Tokens: ['It', 'can', 'not', 'contain', 'NULL', 'values', '.']
  Lemmas: ['it', 'can', 'not', 'contain', 'NULL', 'value', '.']
  POS:    ['PRON', 'AUX', 'PART', 'VERB', 'PROPN', 'NOUN', 'PUNCT']



## 6. Summary

Stage 3 Base ASAG Preprocessing is fully verified:
- [x] Rule-based sentence segmentation (`sentencizer`) without running the dependency parser.
- [x] Standardized tokenization and non-destructive normalization.
- [x] Canonical lemmatization (`identifies` $\rightarrow$ `identify`).
- [x] Universal POS and Penn Treebank tagging.
- [x] Full preservation of syntactic structure, negation, and punctuation for Stage 4 (Dependency Parsing).